In [2]:
# in this script, arcGIS is accessed and feature layer is updated
import lightgbm as lgb
import pandas as pd
import numpy as np
import boto3
import geopandas as gpd
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

In [3]:
s3 = boto3.client('s3')
df = pd.read_parquet(f's3://processed-data-809918852303-us-east-1-an/merged_maxt_DEPLOYMENT.parquet')

In [5]:
# retrieve LGBM text file.....
var='maxt'
model_maxt_thresh_1 = lgb.Booster(model_file=f'lgbm_production_model_{var}_2F_window_nozones.txt')
model_maxt_thresh_2 = lgb.Booster(model_file=f'lgbm_production_model_{var}_3F_window_nozones.txt')
model_maxt_thresh_3 = lgb.Booster(model_file=f'lgbm_production_model_{var}_4F_window.txt')
model_maxt_thresh_4 = lgb.Booster(model_file=f'lgbm_production_model_{var}_5F_window.txt')

var='mint'
model_mint_thresh_1 = lgb.Booster(model_file=f'lgbm_production_model_{var}_2F_window.txt')
model_mint_thresh_2 = lgb.Booster(model_file=f'lgbm_production_model_{var}_3F_window.txt')
model_mint_thresh_3 = lgb.Booster(model_file=f'lgbm_production_model_{var}_4F_window.txt')
model_mint_thresh_4 = lgb.Booster(model_file=f'lgbm_production_model_{var}_5F_window.txt')

var='qpf'
model_qpf_thresh_1 = lgb.Booster(model_file=f'lgbm_production_model_{var}_0.25_window.txt')
model_qpf_thresh_2 = lgb.Booster(model_file=f'lgbm_production_model_{var}_0.75_window.txt')

In [28]:
import datetime # get current date
date = datetime.date.today()

2026-09-07


In [10]:
s3 = boto3.client('s3')
df = pd.read_parquet(f's3://processed-data-809918852303-us-east-1-an/merged_maxt_DEPLOYMENT.parquet')

df = df[df['model_run_date'] == date]

df = df.drop(columns=['state', 'zone_id', 'name'])
df = df.sort_values(by="date").reset_index(drop=True)

# extracting year for each row
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# extracting total weeks for each year
max_weeks = df.groupby('year')['week'].transform('max')
max_weeks = np.where(max_weeks.isin([52, 53]), max_weeks, 52)

# decomposing discrete week into continuous sin/cos component
df['week_sin'] = np.sin(2 * np.pi * df['week'] / max_weeks)
df['week_cos'] = np.cos(2 * np.pi * df['week'] / max_weeks)

df = df.drop(columns=['year'])

X_features_maxt = df[['public_zone', 'week_sin', 'week_cos', 'forecast_day', 'nbm_minus_obs_window_7d', 'nbm_minus_obs_window_60d']] # matching features
X_features_maxt2 = df[['week_sin', 'week_cos', 'forecast_day', 'nbm_minus_obs_window_7d', 'nbm_minus_obs_window_60d']] # matching features

In [12]:
s3 = boto3.client('s3')
df = pd.read_parquet(f's3://processed-data-809918852303-us-east-1-an/merged_mint_DEPLOYMENT.parquet')

df = df[df['model_run_date'] == date]

df = df.drop(columns=['state', 'zone_id', 'name'])
df = df.sort_values(by="date").reset_index(drop=True)

# extracting year for each row
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# extracting total weeks for each year
max_weeks = df.groupby('year')['week'].transform('max')
max_weeks = np.where(max_weeks.isin([52, 53]), max_weeks, 52)

# decomposing discrete week into continuous sin/cos component
df['week_sin'] = np.sin(2 * np.pi * df['week'] / max_weeks)
df['week_cos'] = np.cos(2 * np.pi * df['week'] / max_weeks)

df = df.drop(columns=['year'])

X_features_mint = df[['public_zone', 'week_sin', 'week_cos', 'forecast_day', 'nbm_minus_obs_window_7d', 'nbm_minus_obs_window_60d']] # matching features

In [13]:
s3 = boto3.client('s3')
df = pd.read_parquet(f's3://processed-data-809918852303-us-east-1-an/merged_qpf24pmean_DEPLOYMENT.parquet')

df = df[df['model_run_date'] == date]

df = df.drop(columns=['state', 'zone_id', 'name'])
df = df.sort_values(by="date").reset_index(drop=True)

# extracting year for each row
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# extracting total weeks for each year
max_weeks = df.groupby('year')['week'].transform('max')
max_weeks = np.where(max_weeks.isin([52, 53]), max_weeks, 52)

# decomposing discrete week into continuous sin/cos component
df['week_sin'] = np.sin(2 * np.pi * df['week'] / max_weeks)
df['week_cos'] = np.cos(2 * np.pi * df['week'] / max_weeks)

df = df.drop(columns=['year'])

X_features_qpf = df[['public_zone', 'week_sin', 'week_cos', 'forecast_day', 'nbm_minus_obs_window_7d', 'nbm_minus_obs_window_60d']] # matching features

In [15]:
# run predictions on all var for all thresh
preds_maxt1 = model_maxt_thresh_1.predict(X_features_maxt2)  
preds_maxt2 = model_maxt_thresh_2.predict(X_features_maxt2)  
preds_maxt3 = model_maxt_thresh_3.predict(X_features_maxt)  
preds_maxt4 = model_maxt_thresh_4.predict(X_features_maxt)  

preds_mint1 = model_mint_thresh_1.predict(X_features_mint)  
preds_mint2 = model_mint_thresh_2.predict(X_features_mint)  
preds_mint3 = model_mint_thresh_3.predict(X_features_mint)  
preds_mint4 = model_mint_thresh_4.predict(X_features_mint)  

preds_qpf1 = model_qpf_thresh_1.predict(X_features_qpf)
preds_qpf2 = model_qpf_thresh_2.predict(X_features_qpf)

argmax_maxt1 = np.argmax(preds_maxt1, axis=1)
argmax_maxt2 = np.argmax(preds_maxt2, axis=1)
argmax_maxt3 = np.argmax(preds_maxt3, axis=1)
argmax_maxt4 = np.argmax(preds_maxt4, axis=1)

argmax_mint1 = np.argmax(preds_maxt1, axis=1)
argmax_mint2 = np.argmax(preds_maxt2, axis=1)
argmax_mint3 = np.argmax(preds_maxt3, axis=1)
argmax_mint4 = np.argmax(preds_maxt4, axis=1)

argmax_qpf1 = np.argmax(preds_qpf1, axis=1)
argmax_qpf2 = np.argmax(preds_qpf2, axis=1)

# 2. Extract every class column using slice notation [:, column_index]
df_predictions = pd.DataFrame({
    'public_zone': df['public_zone'],
    'model_run_date': df['model_run_date'],
    'forecast_day': df['forecast_day'],
    'date': df['date'],
    
    'maxt1_predicted_class': argmax_maxt1,
    'maxt1_prob_class0': preds_maxt1[:, 0],  # reasonable forecast
    'maxt1_prob_class1': preds_maxt1[:, 1],  # likely overforecast
    'maxt1_prob_class2': preds_maxt1[:, 2],  # likely underforecast
    
    'maxt2_predicted_class': argmax_maxt2,
    'maxt2_prob_class0': preds_maxt2[:, 0],
    'maxt2_prob_class1': preds_maxt2[:, 1],
    'maxt2_prob_class2': preds_maxt2[:, 2],

    'maxt3_predicted_class': argmax_maxt3,
    'maxt3_prob_class0': preds_maxt3[:, 0],
    'maxt3_prob_class1': preds_maxt3[:, 1],
    'maxt3_prob_class2': preds_maxt3[:, 2],
    
    'maxt4_predicted_class': argmax_maxt4,
    'maxt4_prob_class0': preds_maxt4[:, 0],
    'maxt4_prob_class1': preds_maxt4[:, 1],
    'maxt4_prob_class2': preds_maxt4[:, 2],

    'mint1_predicted_class': argmax_mint1,
    'mint1_prob_class0': preds_mint1[:, 0],  # reasonable forecast
    'mint1_prob_class1': preds_mint1[:, 1],  # likely overforecast
    'mint1_prob_class2': preds_mint1[:, 2],  # likely underforecast etc etc..
    
    'mint2_predicted_class': argmax_mint2,
    'mint2_prob_class0': preds_mint2[:, 0],
    'mint2_prob_class1': preds_mint2[:, 1],
    'mint2_prob_class2': preds_mint2[:, 2],

    'mint3_predicted_class': argmax_mint3,
    'mint3_prob_class0': preds_mint3[:, 0],
    'mint3_prob_class1': preds_mint3[:, 1],
    'mint3_prob_class2': preds_mint3[:, 2],
    
    'mint4_predicted_class': argmax_mint4,
    'mint4_prob_class0': preds_mint4[:, 0],
    'mint4_prob_class1': preds_mint4[:, 1],
    'mint4_prob_class2': preds_mint4[:, 2],
    
    'qpf1_predicted_class': argmax_qpf1,
    'qpf1_prob_class0': preds_qpf1[:, 0],
    'qpf1_prob_class1': preds_qpf1[:, 1],
    'qpf1_prob_class2': preds_qpf1[:, 2],
    
    'qpf2_predicted_class': argmax_qpf2,
    'qpf2_prob_class0': preds_qpf2[:, 0],
    'qpf2_prob_class1': preds_qpf2[:, 1],
    'qpf2_prob_class2': preds_qpf2[:, 2],
})

In [16]:
df_predictions

,public_zone,model_run_date,forecast_day,date,maxt1_predicted_class,maxt1_prob_class0,maxt1_prob_class1,maxt1_prob_class2,maxt2_predicted_class,maxt2_prob_class0,...,mint4_prob_class1,mint4_prob_class2,qpf1_predicted_class,qpf1_prob_class0,qpf1_prob_class1,qpf1_prob_class2,qpf2_predicted_class,qpf2_prob_class0,qpf2_prob_class1,qpf2_prob_class2
0,1,2026-09-06,1,2026-09-06,0,0.635349,0.195648,0.169002,0,0.715698,...,0.136085,0.046979,2,0.335785,0.048718,0.615497,0,0.519683,0.000090,0.480227
1,2354,2026-09-06,1,2026-09-06,0,0.501349,0.235883,0.262768,0,0.553321,...,0.111069,0.132480,0,0.812773,0.000447,0.186780,0,0.942659,0.000063,0.057278
2,739,2026-09-06,1,2026-09-06,0,0.642986,0.198456,0.158558,0,0.716140,...,0.092779,0.032623,1,0.012901,0.805048,0.182051,1,0.012235,0.901149,0.086617
3,2355,2026-09-06,1,2026-09-06,0,0.498773,0.239631,0.261596,0,0.555942,...,0.120489,0.129850,0,0.812351,0.000285,0.187364,0,0.942807,0.000063,0.057129
4,2356,2026-09-06,1,2026-09-06,0,0.502649,0.233728,0.263623,0,0.553810,...,0.120315,0.131102,0,0.809416,0.000385,0.190199,0,0.942973,0.000063,0.056964
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26945,2240,2026-09-06,7,2026-09-12,2,0.228440,0.360568,0.410992,2,0.222875,...,0.235549,0.477223,1,0.052793,0.710456,0.236750,2,0.309832,0.001386,0.688783
26946,2239,2026-09-06,7,2026-09-12,2,0.227546,0.372035,0.400419,2,0.219795,...,0.239172,0.439430,1,0.047954,0.715547,0.236499,2,0.310302,0.000750,0.688949
26947,2238,2026-09-06,7,2026-09-12,2,0.227853,0.371189,0.400958,2,0.219502,...,0.240889,0.438439,1,0.060003,0.697184,0.242814,2,0.347238,0.000734,0.652028
26948,2262,2026-09-06,7,2026-09-12,2,0.234756,0.362581,0.402664,2,0.226468,...,0.217745,0.499147,1,0.073865,0.613791,0.312344,2,0.304981,0.000765,0.694254


In [17]:
# get columns of highest probabilities for confidence mapping later on
df_predictions['maxt1_confidence'] = df_predictions[['maxt1_prob_class0', 'maxt1_prob_class1', 'maxt1_prob_class2']].max(axis=1)
df_predictions['maxt2_confidence'] = df_predictions[['maxt2_prob_class0', 'maxt2_prob_class1', 'maxt2_prob_class2']].max(axis=1)
df_predictions['maxt3_confidence'] = df_predictions[['maxt3_prob_class0', 'maxt3_prob_class1', 'maxt3_prob_class2']].max(axis=1)
df_predictions['maxt4_confidence'] = df_predictions[['maxt4_prob_class0', 'maxt4_prob_class1', 'maxt4_prob_class2']].max(axis=1)

df_predictions['mint1_confidence'] = df_predictions[['mint1_prob_class0', 'mint1_prob_class1', 'mint1_prob_class2']].max(axis=1)
df_predictions['mint2_confidence'] = df_predictions[['mint2_prob_class0', 'mint2_prob_class1', 'mint2_prob_class2']].max(axis=1)
df_predictions['mint3_confidence'] = df_predictions[['mint3_prob_class0', 'mint3_prob_class1', 'mint3_prob_class2']].max(axis=1)
df_predictions['mint4_confidence'] = df_predictions[['mint4_prob_class0', 'mint4_prob_class1', 'mint4_prob_class2']].max(axis=1)

df_predictions['qpf1_confidence'] = df_predictions[['qpf1_prob_class0', 'qpf1_prob_class1', 'qpf1_prob_class2']].max(axis=1)
df_predictions['qpf2_confidence'] = df_predictions[['qpf2_prob_class0', 'qpf2_prob_class1', 'qpf2_prob_class2']].max(axis=1)

In [18]:
# check to make sure logic is correct!
columns = ['maxt2_prob_class0', 'maxt2_prob_class1', 'maxt2_prob_class2', 'maxt2_confidence']
df_predictions[columns][0::7].iloc[500:510]

,maxt2_prob_class0,maxt2_prob_class1,maxt2_prob_class2,maxt2_confidence
3500,0.654710,0.195136,0.150154,0.654710
3507,0.599732,0.227524,0.172745,0.599732
3514,0.597943,0.231146,0.170911,0.597943
3521,0.605387,0.215813,0.178800,0.605387
3528,0.563447,0.220874,0.215679,0.563447
3535,0.714608,0.171530,0.113862,0.714608
3542,0.698893,0.166904,0.134203,0.698893
3549,0.715729,0.169075,0.115196,0.715729
3556,0.617867,0.214286,0.167847,0.617867
3563,0.575489,0.203354,0.221157,0.575489


Getting training set from merged df

In [19]:
# load raw spatial data to append to test set
gdf_map = gpd.read_file("https://www.weather.gov/source/gis/Shapefiles/WSOM/z_16ap26.zip")

values_to_drop = ['AK', 'AS', 'FM', 'GU', 'HI', 'MH', 'MP', 'PR', 'PW', 'VI']
gdf_map = gdf_map[~gdf_map['STATE'].isin(values_to_drop)]

gdf_map['unique_zone_str'] = gdf_map['STATE'] + '_' + gdf_map['ZONE']
gdf_map['public_zone'] = gdf_map['unique_zone_str'].astype('category').cat.codes + 1

gdf_map = gdf_map.dissolve(by='unique_zone_str', as_index=False)
gdf_map = gdf_map.to_crs(epsg=4269)

In [20]:
gdf_map = gdf_map.drop(columns=[
 'CWA',
 'TIME_ZONE',
 'FE_AREA',
 'ZONE',
 'STATE_ZONE',
 'SHORTNAME'])

In [21]:
gdf = gdf_map.merge(df_predictions, on='public_zone', how='inner')

In [22]:
gdf

,unique_zone_str,geometry,STATE,NAME,LON,LAT,public_zone,model_run_date,forecast_day,date,...,maxt1_confidence,maxt2_confidence,maxt3_confidence,maxt4_confidence,mint1_confidence,mint2_confidence,mint3_confidence,mint4_confidence,qpf1_confidence,qpf2_confidence
0,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06,1,2026-09-06,...,0.635349,0.715698,0.848764,0.877169,0.590533,0.669246,0.757166,0.816936,0.615497,0.519683
1,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06,2,2026-09-07,...,0.551170,0.617724,0.764316,0.804746,0.623410,0.698788,0.762934,0.825987,0.494692,0.745240
2,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06,3,2026-09-08,...,0.457692,0.485039,0.639698,0.666141,0.563699,0.641147,0.701665,0.786383,0.598700,0.594217
3,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06,4,2026-09-09,...,0.404909,0.423530,0.578530,0.625082,0.539424,0.606408,0.666207,0.748274,0.627437,0.574824
4,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06,5,2026-09-10,...,0.375057,0.393930,0.525634,0.555038,0.447475,0.500004,0.536959,0.623665,0.709064,0.840429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26945,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06,3,2026-09-08,...,0.380940,0.400786,0.428146,0.470348,0.419128,0.471329,0.549577,0.570492,0.929027,0.983113
26946,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06,4,2026-09-09,...,0.368499,0.386141,0.489567,0.465025,0.378920,0.422622,0.448293,0.466500,0.899914,0.980138
26947,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06,5,2026-09-10,...,0.364738,0.387776,0.406180,0.405235,0.423921,0.428528,0.377968,0.386467,0.814561,0.965616
26948,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06,6,2026-09-11,...,0.397095,0.413655,0.527108,0.529186,0.381025,0.376936,0.375659,0.477083,0.664074,0.925834


In [23]:
# append 12 hours ahead for arcgis purposes
gdf['date'] = gdf['date'] + pd.Timedelta(hours=12)
gdf['model_run_date'] = gdf['model_run_date'] + pd.Timedelta(hours=12)

In [24]:
gdf

,unique_zone_str,geometry,STATE,NAME,LON,LAT,public_zone,model_run_date,forecast_day,date,...,maxt1_confidence,maxt2_confidence,maxt3_confidence,maxt4_confidence,mint1_confidence,mint2_confidence,mint3_confidence,mint4_confidence,qpf1_confidence,qpf2_confidence
0,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06 12:00:00,1,2026-09-06 12:00:00,...,0.635349,0.715698,0.848764,0.877169,0.590533,0.669246,0.757166,0.816936,0.615497,0.519683
1,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06 12:00:00,2,2026-09-07 12:00:00,...,0.551170,0.617724,0.764316,0.804746,0.623410,0.698788,0.762934,0.825987,0.494692,0.745240
2,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06 12:00:00,3,2026-09-08 12:00:00,...,0.457692,0.485039,0.639698,0.666141,0.563699,0.641147,0.701665,0.786383,0.598700,0.594217
3,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06 12:00:00,4,2026-09-09 12:00:00,...,0.404909,0.423530,0.578530,0.625082,0.539424,0.606408,0.666207,0.748274,0.627437,0.574824
4,AL_001,"POLYGON ((-87.9853 35.00591, -87.8812 35.00571...",AL,Lauderdale,-87.6543,34.9015,1,2026-09-06 12:00:00,5,2026-09-10 12:00:00,...,0.375057,0.393930,0.525634,0.555038,0.447475,0.500004,0.536959,0.623665,0.709064,0.840429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26945,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06 12:00:00,3,2026-09-08 12:00:00,...,0.380940,0.400786,0.428146,0.470348,0.419128,0.471329,0.549577,0.570492,0.929027,0.983113
26946,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06 12:00:00,4,2026-09-09 12:00:00,...,0.368499,0.386141,0.489567,0.465025,0.378920,0.422622,0.448293,0.466500,0.899914,0.980138
26947,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06 12:00:00,5,2026-09-10 12:00:00,...,0.364738,0.387776,0.406180,0.405235,0.423921,0.428528,0.377968,0.386467,0.814561,0.965616
26948,WY_199,"POLYGON ((-107.3756 45.00121, -107.351 45.0014...",WY,Sheridan Foothills,-106.6472,44.7977,3850,2026-09-06 12:00:00,6,2026-09-11 12:00:00,...,0.397095,0.413655,0.527108,0.529186,0.381025,0.376936,0.375659,0.477083,0.664074,0.925834


In [27]:
# get feature layer item using feature layer item ID
item_id = "ce78b0087377409ca8eb3b9caef40117"
feature_layer_item = gis.content.get(item_id)

# save updated df to same geojson (MUST BE SAME NAME !!)
gjson_path = f"daily_update.geojson"
gdf.to_file(gjson_path, driver="GeoJSON")

# access the feature layer  collection manager, this is where we overwrite
flc = FeatureLayerCollection.fromitem(feature_layer_item)
flc.manager.overwrite(gjson_path)

print("feature layer overwritten ,,")

feature layer overwritten ,,
